In [1]:
import re
import pandas as pd
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt

In [50]:
with open("../data/raw/all_genres_chart_source.html", "r", encoding="utf-8") as f:
    html = f.read()

print(html[:500])


<!doctype html>
<head>
	
	<!-- Google tag (gtag.js) -->
<script async src="https://www.googletagmanager.com/gtag/js?id=G-TV9F68FR5Y" type="eabc8260c164f3f2546fd7d9-text/javascript"></script>
<script type="eabc8260c164f3f2546fd7d9-text/javascript">
  window.dataLayer = window.dataLayer || [];
  function gtag(){dataLayer.push(arguments);}
  gtag('js', new Date());

  gtag('config', 'G-TV9F68FR5Y');
</script>


	
	
<meta charset="UTF-8">
<meta name="DESCRIPTION" content="Rankings for Artists, Trac


In [51]:
soup = BeautifulSoup(html, "lxml")

print("TITLE:", soup.title.text if soup.title else "No title found")

rows = soup.find_all("div", id="top10artistchart-full")
print("Main chart rows found:", len(rows))

TITLE: BeatStats - Top Tracks
Main chart rows found: 100


In [52]:
print(rows[0].prettify()[:2500])

<div id="top10artistchart-full">
 <div id="top10artistchart-number">
  1
 </div>
 <div id="top10artistchart-image">
  <img height="50" src="/images/track/small/20625808.jpg?version=27739"/>
 </div>
 <div id="top10trackchart-text-large">
  <div id="top10trackchart-artistname">
   JONAS BLUE, MALIVE
  </div>
  <div id="top10trackchart-title">
   <strong>
    Edge of Desire
   </strong>
  </div>
  <div id="top10trackchart-points">
   <strong>
    21094
   </strong>
   POINTS /
   <strong>
    285
   </strong>
   DAYS
   <div id="genreplacement">
    <span class="curved-box" style="background-color: #8d4654;">
     HOUSE
    </span>
   </div>
  </div>
  <a href="https://www.beatport.com/track/edge-of-desire/20625808" target="_blank">
   <div id="shopping-icon-full">
    <img height="20" src="/_images/interface/shopping_icon_alpha.png" width="20"/>
   </div>
  </a>
  <a data-cf-modified-eabc8260c164f3f2546fd7d9-="" href="#" id="20625808" onclick="if (!window.__cfRLUnblockHandlers) return fa

In [53]:
def parse_points_days(text):
    points_match = re.search(r"(\d+)\s+POINTS", text)
    days_match = re.search(r"(\d+)\s+DAYS", text)

    points = int(points_match.group(1)) if points_match else None
    days = int(days_match.group(1)) if days_match else None

    return points, days

In [54]:
def parse_artists(artist_text):
    if not artist_text:
        return []
    return [a.strip() for a in artist_text.split(",") if a.strip()]

In [55]:
def parse_genre_from_row(row):
    genre_div = row.find("div", id="genreplacement")
    if genre_div:
        genre_text = genre_div.get_text(" ", strip=True)
        if genre_text:
            return genre_text

    for tag in row.find_all(["div", "span"]):
        tag_id = (tag.get("id") or "").lower()
        tag_classes = " ".join(tag.get("class", [])).lower()
        text = tag.get_text(" ", strip=True)

        if ("genre" in tag_id or "genre" in tag_classes) and text:
            return text

    return ""

In [56]:
chart_data = []

for row in rows:
    rank_tag = row.find("div", id="top10artistchart-number")
    artist_tag = row.find("div", id="top10trackchart-artistname")
    title_tag = row.find("div", id="top10trackchart-title")
    points_tag = row.find("div", id="top10trackchart-points")

    # Rank + rise amount
    rank_text = rank_tag.get_text(" ", strip=True) if rank_tag else ""
    numbers = re.findall(r"\d+", rank_text)

    rank = int(numbers[0]) if len(numbers) >= 1 else None
    rise_amount = int(numbers[1]) if len(numbers) >= 2 else 0

    # Other fields
    artist_text = artist_tag.get_text(strip=True) if artist_tag else ""
    title = title_tag.get_text(strip=True) if title_tag else ""
    points_text = points_tag.get_text(" ", strip=True) if points_tag else ""

    points, days = parse_points_days(points_text)
    artists = parse_artists(artist_text)
    genre = parse_genre_from_row(row)

    beatport_link = ""
    for a in row.find_all("a", href=True):
        href = a["href"]
        if "beatport.com/track/" in href:
            beatport_link = href
            break

    chart_data.append({
        "rank": rank,
        "rise_amount": rise_amount,
        "is_rising": rise_amount > 0,
        "artist_raw": artist_text,
        "artists": artists,
        "num_artists": len(artists),
        "title": title,
        "genre": genre,
        "points": points,
        "days": days,
        "beatport_link": beatport_link
    })

len(chart_data), chart_data[:10]

(100,
 [{'rank': 1,
   'rise_amount': 0,
   'is_rising': False,
   'artist_raw': 'JONAS BLUE, MALIVE',
   'artists': ['JONAS BLUE', 'MALIVE'],
   'num_artists': 2,
   'title': 'Edge of Desire',
   'genre': 'HOUSE',
   'points': 21094,
   'days': 285,
   'beatport_link': 'https://www.beatport.com/track/edge-of-desire/20625808'},
  {'rank': 2,
   'rise_amount': 0,
   'is_rising': False,
   'artist_raw': 'TINASHE, DISCO LINES',
   'artists': ['TINASHE', 'DISCO LINES'],
   'num_artists': 2,
   'title': 'No Broke Boys',
   'genre': 'DANCE / POP',
   'points': 17922,
   'days': 258,
   'beatport_link': 'https://www.beatport.com/track/no-broke-boys/20601832'},
  {'rank': 3,
   'rise_amount': 0,
   'is_rising': False,
   'artist_raw': 'MAU P',
   'artists': ['MAU P'],
   'num_artists': 1,
   'title': 'Like I Like It',
   'genre': 'TECH HOUSE',
   'points': 16951,
   'days': 251,
   'beatport_link': 'https://www.beatport.com/track/like-i-like-it/20505201'},
  {'rank': 4,
   'rise_amount': 0,
  

In [57]:
df = pd.DataFrame(chart_data)
df = df.drop_duplicates(subset=["rank", "artist_raw", "title"])
df = df.sort_values("rank").reset_index(drop=True)

df.head(10)

,rank,rise_amount,is_rising,artist_raw,artists,num_artists,title,genre,points,days,beatport_link
0,1,0,False,"JONAS BLUE, MALIVE","[JONAS BLUE, MALIVE]",2,Edge of Desire,HOUSE,21094,285,https://www.beatport.com/track/edge-of-desire/...
1,2,0,False,"TINASHE, DISCO LINES","[TINASHE, DISCO LINES]",2,No Broke Boys,DANCE / POP,17922,258,https://www.beatport.com/track/no-broke-boys/2...
2,3,0,False,MAU P,[MAU P],1,Like I Like It,TECH HOUSE,16951,251,https://www.beatport.com/track/like-i-like-it/...
3,4,0,False,"HUGEL, SOLTO (FR)","[HUGEL, SOLTO (FR)]",2,Jamaican (Bam Bam),AFRO HOUSE,16248,173,https://www.beatport.com/track/jamaican-bam-ba...
4,5,3,True,JAMBACK,[JAMBACK],1,Positive,MINIMAL / DEEP TECH,15176,173,https://www.beatport.com/track/positive/22236272
5,6,0,False,TOMAN,[TOMAN],1,Verano En NY,TECH HOUSE,13549,228,https://www.beatport.com/track/verano-en-ny/20...
6,7,0,False,"CID, TAYLR RENEE","[CID, TAYLR RENEE]",2,Fancy $hit,TECH HOUSE,13304,170,https://www.beatport.com/track/fancy-hit/21309496
7,8,0,False,PROSPA,[PROSPA],1,Don't Stop,TECH HOUSE,12579,237,https://www.beatport.com/track/dont-stop/20311844
8,9,0,False,"PROSPA, KOSMO KINT","[PROSPA, KOSMO KINT]",2,Love Songs (feat. Kosmo Kint),HOUSE,12178,174,https://www.beatport.com/track/love-songs-feat...
9,10,1,True,"DA HOOL, CASSIAN, YOTTO","[DA HOOL, CASSIAN, YOTTO]",3,Love Parade,MELODIC HOUSE & TECHNO,11337,154,https://www.beatport.com/track/love-parade/217...


In [58]:
df["rise_str"] = df["rise_amount"].apply(lambda x: f"+{x}" if x > 0 else "0")

df[["rank", "rise_amount", "rise_str", "is_rising", "artist_raw", "title", "genre"]].head(20)

,rank,rise_amount,rise_str,is_rising,artist_raw,title,genre
0,1,0,0,False,"JONAS BLUE, MALIVE",Edge of Desire,HOUSE
1,2,0,0,False,"TINASHE, DISCO LINES",No Broke Boys,DANCE / POP
2,3,0,0,False,MAU P,Like I Like It,TECH HOUSE
3,4,0,0,False,"HUGEL, SOLTO (FR)",Jamaican (Bam Bam),AFRO HOUSE
4,5,3,+3,True,JAMBACK,Positive,MINIMAL / DEEP TECH
5,6,0,0,False,TOMAN,Verano En NY,TECH HOUSE
6,7,0,0,False,"CID, TAYLR RENEE",Fancy $hit,TECH HOUSE
7,8,0,0,False,PROSPA,Don't Stop,TECH HOUSE
8,9,0,0,False,"PROSPA, KOSMO KINT",Love Songs (feat. Kosmo Kint),HOUSE
9,10,1,+1,True,"DA HOOL, CASSIAN, YOTTO",Love Parade,MELODIC HOUSE & TECHNO


In [59]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

df

,rank,rise_amount,is_rising,artist_raw,artists,num_artists,title,genre,points,days,beatport_link,rise_str
0,1,0,False,"JONAS BLUE, MALIVE","[JONAS BLUE, MALIVE]",2,Edge of Desire,HOUSE,21094,285,https://www.beatport.com/track/edge-of-desire/...,0
1,2,0,False,"TINASHE, DISCO LINES","[TINASHE, DISCO LINES]",2,No Broke Boys,DANCE / POP,17922,258,https://www.beatport.com/track/no-broke-boys/2...,0
2,3,0,False,MAU P,[MAU P],1,Like I Like It,TECH HOUSE,16951,251,https://www.beatport.com/track/like-i-like-it/...,0
3,4,0,False,"HUGEL, SOLTO (FR)","[HUGEL, SOLTO (FR)]",2,Jamaican (Bam Bam),AFRO HOUSE,16248,173,https://www.beatport.com/track/jamaican-bam-ba...,0
4,5,3,True,JAMBACK,[JAMBACK],1,Positive,MINIMAL / DEEP TECH,15176,173,https://www.beatport.com/track/positive/22236272,+3
5,6,0,False,TOMAN,[TOMAN],1,Verano En NY,TECH HOUSE,13549,228,https://www.beatport.com/track/verano-en-ny/20...,0
6,7,0,False,"CID, TAYLR RENEE","[CID, TAYLR RENEE]",2,Fancy $hit,TECH HOUSE,13304,170,https://www.beatport.com/track/fancy-hit/21309496,0
7,8,0,False,PROSPA,[PROSPA],1,Don't Stop,TECH HOUSE,12579,237,https://www.beatport.com/track/dont-stop/20311844,0
8,9,0,False,"PROSPA, KOSMO KINT","[PROSPA, KOSMO KINT]",2,Love Songs (feat. Kosmo Kint),HOUSE,12178,174,https://www.beatport.com/track/love-songs-feat...,0
9,10,1,True,"DA HOOL, CASSIAN, YOTTO","[DA HOOL, CASSIAN, YOTTO]",3,Love Parade,MELODIC HOUSE & TECHNO,11337,154,https://www.beatport.com/track/love-parade/217...,+1


In [16]:
df["movement_str"] = df["movement"].apply(lambda x: f"+{x}" if x > 0 else str(x))

df[["rank", "movement", "movement_str", "movement_direction", "artist_raw", "title", "genre"]].head(20)

,rank,movement,movement_str,movement_direction,artist_raw,title,genre
0,1,0,0,flat,"JONAS BLUE, MALIVE",Edge of Desire,HOUSE
1,2,0,0,flat,"TINASHE, DISCO LINES",No Broke Boys,DANCE / POP
2,3,0,0,flat,MAU P,Like I Like It,TECH HOUSE
3,4,0,0,flat,"HUGEL, SOLTO (FR)",Jamaican (Bam Bam),AFRO HOUSE
4,5,3,+3,up,JAMBACK,Positive,MINIMAL / DEEP TECH
5,6,0,0,flat,TOMAN,Verano En NY,TECH HOUSE
6,7,0,0,flat,"CID, TAYLR RENEE",Fancy $hit,TECH HOUSE
7,8,0,0,flat,PROSPA,Don't Stop,TECH HOUSE
8,9,0,0,flat,"PROSPA, KOSMO KINT",Love Songs (feat. Kosmo Kint),HOUSE
9,10,1,+1,up,"DA HOOL, CASSIAN, YOTTO",Love Parade,MELODIC HOUSE & TECHNO


In [17]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

df

,rank,movement,movement_direction,artist_raw,artists,title,genre,points,days,beatport_link,movement_str
0,1,0,flat,"JONAS BLUE, MALIVE","[JONAS BLUE, MALIVE]",Edge of Desire,HOUSE,21094,285,https://www.beatport.com/track/edge-of-desire/...,0
1,2,0,flat,"TINASHE, DISCO LINES","[TINASHE, DISCO LINES]",No Broke Boys,DANCE / POP,17922,258,https://www.beatport.com/track/no-broke-boys/2...,0
2,3,0,flat,MAU P,[MAU P],Like I Like It,TECH HOUSE,16951,251,https://www.beatport.com/track/like-i-like-it/...,0
3,4,0,flat,"HUGEL, SOLTO (FR)","[HUGEL, SOLTO (FR)]",Jamaican (Bam Bam),AFRO HOUSE,16248,173,https://www.beatport.com/track/jamaican-bam-ba...,0
4,5,3,up,JAMBACK,[JAMBACK],Positive,MINIMAL / DEEP TECH,15176,173,https://www.beatport.com/track/positive/22236272,+3
5,6,0,flat,TOMAN,[TOMAN],Verano En NY,TECH HOUSE,13549,228,https://www.beatport.com/track/verano-en-ny/20...,0
6,7,0,flat,"CID, TAYLR RENEE","[CID, TAYLR RENEE]",Fancy $hit,TECH HOUSE,13304,170,https://www.beatport.com/track/fancy-hit/21309496,0
7,8,0,flat,PROSPA,[PROSPA],Don't Stop,TECH HOUSE,12579,237,https://www.beatport.com/track/dont-stop/20311844,0
8,9,0,flat,"PROSPA, KOSMO KINT","[PROSPA, KOSMO KINT]",Love Songs (feat. Kosmo Kint),HOUSE,12178,174,https://www.beatport.com/track/love-songs-feat...,0
9,10,1,up,"DA HOOL, CASSIAN, YOTTO","[DA HOOL, CASSIAN, YOTTO]",Love Parade,MELODIC HOUSE & TECHNO,11337,154,https://www.beatport.com/track/love-parade/217...,+1


In [60]:
print("Total tracks:", len(df))
print("Min rank:", df["rank"].min())
print("Max rank:", df["rank"].max())
print("Unique ranks:", df["rank"].nunique())
print("Duplicate ranks:", df["rank"].duplicated().sum())
print("\nMissing values:")
print(df.isnull().sum())

Total tracks: 100
Min rank: 1
Max rank: 100
Unique ranks: 100
Duplicate ranks: 0

Missing values:
rank             0
rise_amount      0
is_rising        0
artist_raw       0
artists          0
num_artists      0
title            0
genre            0
points           0
days             0
beatport_link    0
rise_str         0
dtype: int64


In [61]:
df.describe()

,rank,rise_amount,num_artists,points,days
count,100.000000,100.000000,100.000000,100.000000,100.000000
mean,50.500000,6.500000,1.920000,7275.000000,112.890000
std,29.011492,20.157712,1.011849,3240.017845,46.734344
min,1.000000,0.000000,1.000000,4234.000000,54.000000
25%,25.750000,0.000000,1.000000,5094.750000,82.000000
50%,50.500000,1.000000,2.000000,6310.500000,95.000000
75%,75.250000,3.000000,3.000000,8232.500000,134.000000
max,100.000000,155.000000,6.000000,21094.000000,285.000000


In [62]:
df["genre"].value_counts().head(30)

genre
TECH HOUSE                28
HOUSE                     22
DANCE / POP               14
AFRO HOUSE                11
MELODIC HOUSE & TECHNO     8
UK GARAGE / BASSLINE       4
BASS HOUSE                 3
MINIMAL / DEEP TECH        2
DEEP HOUSE                 2
INDIE DANCE                2
TRANCE (MAIN FLOOR)        2
FUNKY HOUSE                1
MAINSTAGE                  1
Name: count, dtype: int64

In [63]:
df["rise_amount"].value_counts().sort_index()

rise_amount
0      40
1      17
2      10
3      11
4       5
5       5
6       1
8       1
9       1
22      1
26      1
34      1
36      1
42      1
48      1
51      1
98      1
155     1
Name: count, dtype: int64

In [64]:
df.sort_values("rise_amount", ascending=False)[
    ["rank", "rise_str", "artist_raw", "title", "genre"]
].head(15)

,rank,rise_str,artist_raw,title,genre
66,67,+155,"ANOTR, 54 ULTRA",Talk To You,HOUSE
79,80,+98,"RIORDAN, SILVA BUMPA",Lifting,UK GARAGE / BASSLINE
47,48,+51,"REINIER ZONNEVELD, GORDO",Loco Loco,DANCE / POP
36,37,+48,TRACE (UZ),Good Time,HOUSE
35,36,+42,MAU P,neck,TECH HOUSE
42,43,+36,ESSE (US),Make My Day,TECH HOUSE
46,47,+34,JOSHWA,Out of My Mind,TECH HOUSE
69,70,+26,"MILKY, MALL GRAB",Just The Way You Are,HOUSE
20,21,+22,BRUNELLO,Ghost Dance,INDIE DANCE
11,12,+9,"ULTRA NATE, HUGEL",Free (You Got To Live),AFRO HOUSE


In [65]:
df["is_rising"].value_counts()

is_rising
True     60
False    40
Name: count, dtype: int64

In [66]:
df_artists = df.explode("artists").reset_index(drop=True)
df_artists.head(20)

,rank,rise_amount,is_rising,artist_raw,artists,num_artists,title,genre,points,days,beatport_link,rise_str
0,1,0,False,"JONAS BLUE, MALIVE",JONAS BLUE,2,Edge of Desire,HOUSE,21094,285,https://www.beatport.com/track/edge-of-desire/...,0
1,1,0,False,"JONAS BLUE, MALIVE",MALIVE,2,Edge of Desire,HOUSE,21094,285,https://www.beatport.com/track/edge-of-desire/...,0
2,2,0,False,"TINASHE, DISCO LINES",TINASHE,2,No Broke Boys,DANCE / POP,17922,258,https://www.beatport.com/track/no-broke-boys/2...,0
3,2,0,False,"TINASHE, DISCO LINES",DISCO LINES,2,No Broke Boys,DANCE / POP,17922,258,https://www.beatport.com/track/no-broke-boys/2...,0
4,3,0,False,MAU P,MAU P,1,Like I Like It,TECH HOUSE,16951,251,https://www.beatport.com/track/like-i-like-it/...,0
5,4,0,False,"HUGEL, SOLTO (FR)",HUGEL,2,Jamaican (Bam Bam),AFRO HOUSE,16248,173,https://www.beatport.com/track/jamaican-bam-ba...,0
6,4,0,False,"HUGEL, SOLTO (FR)",SOLTO (FR),2,Jamaican (Bam Bam),AFRO HOUSE,16248,173,https://www.beatport.com/track/jamaican-bam-ba...,0
7,5,3,True,JAMBACK,JAMBACK,1,Positive,MINIMAL / DEEP TECH,15176,173,https://www.beatport.com/track/positive/22236272,+3
8,6,0,False,TOMAN,TOMAN,1,Verano En NY,TECH HOUSE,13549,228,https://www.beatport.com/track/verano-en-ny/20...,0
9,7,0,False,"CID, TAYLR RENEE",CID,2,Fancy $hit,TECH HOUSE,13304,170,https://www.beatport.com/track/fancy-hit/21309496,0


In [67]:
artist_counts = df_artists["artists"].value_counts().reset_index()
artist_counts.columns = ["artist", "track_count"]

artist_counts.head(20)

,artist,track_count
0,MAU P,4
1,ADAM PORT,3
2,PROSPA,3
3,SIDEPIECE,3
4,CALVIN HARRIS,3
5,ODD MOB,3
6,MAX STYLER,3
7,SILVA BUMPA,3
8,HUGEL,3
9,CHRIS LAKE,3


In [68]:
artist_summary = (
    df_artists.groupby("artists", as_index=False)
    .agg(
        track_count=("title", "count"),
        avg_points=("points", "mean"),
        avg_days=("days", "mean"),
        rising_tracks=("is_rising", "sum")
    )
    .sort_values(["track_count", "avg_points"], ascending=[False, False])
)

artist_summary.head(20)

,artists,track_count,avg_points,avg_days,rising_tracks
94,MAU P,4,9299.750000,138.500000,1
60,HUGEL,3,11517.000000,147.666667,2
113,PROSPA,3,9801.000000,169.333333,0
29,CHRIS LAKE,3,8076.666667,123.333333,1
129,SIDEPIECE,3,7832.666667,120.666667,3
96,MAX STYLER,3,6682.666667,111.000000,1
24,CALVIN HARRIS,3,6623.333333,104.000000,3
109,ODD MOB,3,5680.000000,84.333333,2
4,ADAM PORT,3,4984.666667,81.666667,1
130,SILVA BUMPA,3,4613.333333,79.333333,3


In [69]:
df.sort_values("num_artists", ascending=False)[
    ["rank", "num_artists", "artist_raw", "title", "genre"]
].head(15)

,rank,num_artists,artist_raw,title,genre
98,99,6,"BOYS NOIZE, &ME, RAMPA, ADAM PORT, KEINEMUSIK,...",Crazy For It (feat. Vinson),MELODIC HOUSE & TECHNO
93,94,5,"&ME, RAMPA, ADAM PORT, KEINEMUSIK, CHUALA",Say What (feat. Chuala),AFRO HOUSE
34,35,5,"MODJO, SPARROW & BARBOSSA, KOSHI, DAYMAAN, SPA...",Lady (Hear Me Tonight) - Remix,AFRO HOUSE
44,45,4,"EMANUELE ESPOSITO, GIANNI ROMANO, HELEN TESFAZ...",It's Not Right,AFRO HOUSE
74,75,3,"SHIMZA, KASANGO, AR/CO",Fire Fire,AFRO HOUSE
30,31,3,"SHAKEDOWN, LAYTON GIORDANI, ANYMA (OFC)",At Night (Anyma x Layton Giordani Extended Remix),MELODIC HOUSE & TECHNO
65,66,3,"MOBY, BLOND:ISH, KIKO FRANCO",Natural Blues,HOUSE
26,27,3,"JOCELYN BROWN, LUKE ALESSI, CHLOé CAILLET",The One,HOUSE
67,68,3,"SONNY FODERA, D.O.D, POPPY BASKCOMB",Think About Us,DANCE / POP
71,72,3,"KASKADE, CID, ANABEL ENGLUND",Vision Blurred,TECH HOUSE


In [70]:
df["num_artists"].value_counts().sort_index()

num_artists
1    42
2    32
3    22
4     1
5     2
6     1
Name: count, dtype: int64

In [30]:
artist_summary = (
    df_artists.groupby("artists", as_index=False)
    .agg(
        track_count=("title", "count"),
        avg_points=("points", "mean"),
        avg_days=("days", "mean"),
        avg_movement=("movement", "mean")
    )
    .sort_values(["track_count", "avg_points"], ascending=[False, False])
)

artist_summary.head(20)

,artists,track_count,avg_points,avg_days,avg_movement
94,MAU P,4,9299.750000,138.500000,10.500000
60,HUGEL,3,11517.000000,147.666667,4.000000
113,PROSPA,3,9801.000000,169.333333,0.000000
29,CHRIS LAKE,3,8076.666667,123.333333,0.333333
129,SIDEPIECE,3,7832.666667,120.666667,1.666667
96,MAX STYLER,3,6682.666667,111.000000,1.333333
24,CALVIN HARRIS,3,6623.333333,104.000000,2.000000
109,ODD MOB,3,5680.000000,84.333333,1.000000
4,ADAM PORT,3,4984.666667,81.666667,1.333333
130,SILVA BUMPA,3,4613.333333,79.333333,34.333333


In [71]:
df_export = df.copy()
df_export["artists"] = df_export["artists"].apply(lambda x: " | ".join(x))

df_export.to_csv("../data/processed/all_genres_chart_clean.csv", index=False)
print("Saved ../data/processed/all_genres_chart_clean.csv")

Saved ../data/processed/all_genres_chart_clean.csv
